# Querychat Customization Experiments

This notebook documents the experiments I ran to decide what to actually put in the M4 Option A implementation.
Three things needed to be figured out:

1. **What to put in `extra_instructions`** — the M3 data_description was really sparse (only 5 columns listed out of 16, no mention of `risk_value` at all). I wanted to see if a richer prompt actually changes how the model answers.
2. **What to do with `on_tool_request`** — the assignment says to intercept tool calls to validate, log, or transform them. Logging was easy. Enforcement via ToolRejectError took a bit to figure out.
3. **What the user-facing control should be** — tried a few options, landed on an Analysis Scope dropdown.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
import querychat
from chatlas import ChatAnthropic, ToolRejectError

load_dotenv()
API_KEY = os.environ.get("ANTHROPIC_API_KEY")

df = pd.read_csv("../data/raw/sales_and_customer_insights.csv", parse_dates=True)
df["risk_value"] = df["Lifetime_Value"] * df["Churn_Probability"]
df["Launch_Date"] = pd.to_datetime(df["Launch_Date"])

print(df.shape)
print(df.columns.tolist())

## Part 1: Does extra_instructions actually help?

I set up two clients — one with the M3 data_description and one with the richer version + extra_instructions.
Same question to both.

In [ ]:
# M3 version (what we had before)
qc_old = querychat.QueryChat(
    df.copy(),
    "Salescope",
    data_description="""
    This is Sales insights dataset
    Columns:
    - Region: Asia, Europe, North America, South America
    - Most_Frequent_Category: Clothing, Electronics, Home, Sports
    - Lifetime_Value: Numerical float
    - Churn_Probability: Float (0 to 1)
    - Retention_Strategy: Discount, Email Campaign, Loyalty Program
    """,
    client=ChatAnthropic(model="claude-sonnet-4-0", api_key=API_KEY),
)
client_old = qc_old.client(tools="query")

In [ ]:
# M4 version
extra = """
You are working inside the Salescope retention dashboard. Users are sales managers, not data scientists.

risk_value = Lifetime_Value * Churn_Probability. This is the dollar amount at risk if a customer churns.
Treat it as the most useful column when prioritizing interventions.

Rough churn thresholds: above 0.7 = high risk, 0.4-0.7 = medium, below 0.4 = low.

When you answer, say what it means for the business (e.g. "this region has $X at risk").
Keep responses short. Suggest which retention strategy fits when it's relevant.
"""

qc_new = querychat.QueryChat(
    df.copy(),
    "Salescope",
    data_description="""
    Salescope customer dataset — 10,000 e-commerce customers.
    Columns: Customer_ID, Product_ID, Transaction_ID, Purchase_Frequency (1-19),
    Average_Order_Value (20-200), Most_Frequent_Category (Clothing/Electronics/Home/Sports),
    Time_Between_Purchases (days), Region (Asia/Europe/North America/South America),
    Churn_Probability (0-1), Lifetime_Value (100-10000 USD), Launch_Date, Peak_Sales_Date,
    Season (Spring/Summer/Fall/Winter), Preferred_Purchase_Times (Morning/Afternoon/Evening),
    Retention_Strategy (Discount/Email Campaign/Loyalty Program),
    risk_value (derived: Lifetime_Value * Churn_Probability)
    """,
    extra_instructions=extra,
    client=ChatAnthropic(model="claude-sonnet-4-0", api_key=API_KEY),
)
client_new = qc_new.client(tools="query")

In [ ]:
q = "Which region has the highest revenue risk?"

print("--- OLD ---")
print(str(client_old.chat(q, echo="none")))

print("\n--- NEW ---")
print(str(client_new.chat(q, echo="none")))

**What I found:**

Old version: doesn't know about `risk_value` since we never mentioned it, so it either invents a way to compute it from scratch or just answers about churn probability. The answer doesn't mention dollar amounts.

New version: immediately uses `risk_value` as the column, gives a dollar figure for revenue at risk, and mentions a retention strategy recommendation at the end.

That's a pretty clear improvement. The extra_instructions change is worth keeping.

In [ ]:
# second test — ask about a column the old version didn't know
q2 = "When do most high-churn customers prefer to shop?"

print("--- OLD ---")
print(str(client_old.chat(q2, echo="none")))

print("\n--- NEW ---")
print(str(client_new.chat(q2, echo="none")))

**What I found:**

The old version sometimes tries to answer this but the model isn't aware that `Preferred_Purchase_Times` exists (it's not in the M3 data_description). It either hallucinates or says the column doesn't exist.

The new version finds `Preferred_Purchase_Times` right away and groups by it correctly.

**Conclusion for Experiment 1:** Adding the full column list to `data_description` + `extra_instructions` with business framing is the right call. We'll keep both.

---

## Part 2: on_tool_request — logging and scope enforcement

The assignment asks us to use `on_tool_request` to validate, log, or transform tool calls.
I first just added logging to see what the model is actually sending, then built the scope guard on top.

In [ ]:
# basic logging — just print every tool call before it runs
client_logged = qc_new.client(tools="query")

def log_call(request):
    args = request.arguments if isinstance(request.arguments, dict) else {}
    sql = args.get("query", "")
    print(f"[tool] {request.name} | sql={sql[:80]!r}")

client_logged.on_tool_request(log_call)

_ = client_logged.chat("How many customers are in each region?", echo="none")
print("done")

The log line shows the tool name is `querychat_query` and the SQL it generates. This is useful for debugging — you can see exactly what SQL the model is writing and catch problems early.

Next: actually enforcing scope. The idea is that different users might want to restrict the AI to only churn questions or only revenue questions. The `ToolRejectError` in chatlas lets the callback cancel a tool call, and the model gets the rejection message back and tells the user.

In [ ]:
# churn-only scope test
client_churn = qc_new.client(tools=("update", "query"))

def churn_guard(request):
    args = request.arguments if isinstance(request.arguments, dict) else {}
    sql = args.get("query", "")
    if sql:
        if not any(t in sql.lower() for t in ("churn", "churn_probability", "retention_strategy")):
            raise ToolRejectError(
                "Churn Focus Only mode is on. Please ask about churn, "
                "retention strategies, or at-risk customers."
            )

client_churn.on_tool_request(churn_guard)

print("=== churn question (should work) ===")
r1 = client_churn.chat("Which region has the highest average churn probability?", echo="none")
print(str(r1)[:300])

In [ ]:
print("=== non-churn question (should be blocked) ===")
r2 = client_churn.chat("What is the most popular product category?", echo="none")
print(str(r2)[:300])

**What I found:**

The churn question goes through fine. The category question gets blocked — the model receives the `ToolRejectError` message and responds to the user something like "I'm currently restricted to churn and retention questions, try asking about churn probability instead." It handles the rejection gracefully which is exactly what we want.

Note: the ToolRejectError message gets passed back to the model as a tool result error, so the model sees it and can respond appropriately. It doesn't just crash or return nothing.

In [ ]:
# same test for revenue-only scope
client_rev = qc_new.client(tools=("update", "query"))

def revenue_guard(request):
    args = request.arguments if isinstance(request.arguments, dict) else {}
    sql = args.get("query", "")
    if sql:
        if not any(t in sql.lower() for t in ("lifetime_value", "average_order_value", "risk_value")):
            raise ToolRejectError(
                "Revenue Focus Only mode is on. Please ask about "
                "Lifetime_Value, Average_Order_Value, or risk_value."
            )

client_rev.on_tool_request(revenue_guard)

print("=== revenue question ===")
r3 = client_rev.chat("What is the average Lifetime Value by region?", echo="none")
print(str(r3)[:300])

print("\n=== churn question in revenue mode ===")
r4 = client_rev.chat("Show customers with churn probability above 0.9", echo="none")
print(str(r4)[:300])

Revenue guard works the same way. The churn question in revenue mode is rejected, the model tells the user.

**One limitation I noticed:** the scope guard checks column names in the SQL string. This means if the model writes a query that references churn indirectly (e.g. `WHERE risk_value > 5000`) in churn-only mode, it might get blocked even though it's somewhat related. The tradeoff is simplicity — the string check is fast and covers most real cases.

---

## Part 3: Choosing the user-facing control

Options I considered:

**A. Verbosity slider (1–5)**
The idea was to inject something like "be very brief" or "be very detailed" into the system prompt based on the slider value. Problem: querychat's system prompt is set at initialization. To change it mid-session you'd have to reinitialize the client and lose conversation history. Not practical.

**B. Response style dropdown (Bullet / Prose / Executive)**
Same problem as verbosity — you'd need to change the system prompt per session, which querychat doesn't support cleanly without re-init.

**C. Analysis Scope dropdown (Full / Churn Only / Revenue Only)**
This one works well because the restriction happens in the `on_tool_request` callback, not in the system prompt. The callback can read a mutable Python dict that gets updated by a Shiny reactive effect whenever the dropdown changes. So you get runtime control without touching the system prompt.

Also, the scope control maps to a real user need — different people in a sales org care about different metrics. A churn analyst doesn't want the AI going off on revenue tangents. A CFO reviewing LTV doesn't need churn recommendations.

**Decision: Option C (Analysis Scope dropdown)** using `on_tool_request` for enforcement.

In [ ]:
# verify the mutable dict approach works (simulates what the Shiny app does)
client_test = qc_new.client(tools=("update", "query"))

_scope = {"value": "full"}

def dynamic_guard(request):
    args = request.arguments if isinstance(request.arguments, dict) else {}
    sql = args.get("query", "")
    scope = _scope["value"]
    print(f"[guard] scope={scope} sql={sql[:60]!r}")
    if scope == "churn_only" and sql:
        if not any(t in sql.lower() for t in ("churn", "churn_probability", "retention_strategy")):
            raise ToolRejectError("Churn Focus Only mode is on.")

client_test.on_tool_request(dynamic_guard)

# ask in full mode — should work
_scope["value"] = "full"
print("--- full mode ---")
r = client_test.chat("How many customers prefer morning shopping?", echo="none")
print(str(r)[:200])

# switch scope mid-conversation — next tool call should be blocked
_scope["value"] = "churn_only"
print("\n--- switched to churn_only ---")
r2 = client_test.chat("What is the most popular product category?", echo="none")
print(str(r2)[:200])

The mutable dict approach works — the first question goes through in full mode, and after switching `_scope['value']` to `churn_only`, the next non-churn question is blocked.

In the Shiny app, the `_scope` dict gets updated by a `@reactive.effect` that reads the dropdown value, so it stays in sync with whatever the user has selected.

---

## Summary

Three changes went into `src/app.py`:

| Feature | Implementation | Why |
|---------|---------------|-----|
| `extra_instructions` | `SALESCOPE_EXTRA_INSTRUCTIONS` constant passed to `QueryChat()` | Baseline prompt missed 11 columns and had no business framing — experiments showed clear improvement |
| `on_tool_request` | `_handle_tool_request` registered on `qc_vals.client` | Logs SQL to console, enforces scope via `ToolRejectError` |
| Scope dropdown | `ui.input_select("ai_scope_mode")` in AI Insights panel | Cleanest way to control LLM behavior at runtime without re-initializing the client |

The spec doc (`docs/AI_INTEGRATION_TESTING.md`) has the full design rationale.